# Import Dependencies

In [39]:
# !pip install mesa --quiet

In [61]:
try:
    import mesa
except:
    %pip install mesa --quiet
    import mesa
import numpy as np
import math
import matplotlib.pyplot as plt
%matplotlib inline

# Helper Functions

In [ ]:
def get_distance(pos_1, pos_2):
    '''
    Calculate the Euclidean distance between two positions

    used in trade.move()
    '''

    x1, y1 = pos_1
    x2, y2 = pos_2

# Resource Classes

In [41]:
class Sugar(mesa.Agent):
    ''' 
    Sugar
        - contains an amount of sugar
        - grows one amount of sugar for each turn
    '''

    def __init__(self, unique_id, model, pos, max_sugar):
        super().__init__(unique_id, model)
        self.initial_pos = pos # pos refers to position. Mesa’s newer versions track self.pos internally, and adding the line self.pos = pos manually creates inconsistency. That's why we have replaced here to self.initial_pos 
        self.amount = max_sugar
        self.max_sugar = max_sugar

    def step(self):
        '''
        Sugar growth function, adds one unit of sugar each step until max amount
        '''
        self.amount = min([self.max_sugar, self.amount+1])

class Spice(mesa.Agent):
    '''
    Spice:
        - contains an amount of spice
        - grows one amount of spice at each turn
    '''

    def __init__(self, unique_id, model, pos, max_spice):
        super().__init__(unique_id, model)
        self.initial_pos = pos
        self.amount = max_spice # This tells us how much spice at a given time step
        self.max_spice = max_spice # This tells the most that can be at a location

    def step(self):
        '''
        Spice growth function, adds one unit of spice each step until max amount
        '''
        self.amount = min([self.max_spice, self.amount+1])

# Trader Class

In [ ]:
class Trader(mesa.Agent):
    '''
    Trader:
        - has a metabolism for sugar and spice
        - harvest and traders sugar and spice to survive and thrive
    '''

    def __init__(self, unique_id, model, pos, moore=False, sugar=0, spice=0, metabolism_sugar=0, metabolism_spice=0, vision=0):
        super().__init__(unique_id, model)

        # self.pos=pos # Removed to avoid duplicating setting the position and avoid Warning message
        self.moore=moore
        self.sugar=sugar
        self.spice=spice
        self.metabolism_sugar=metabolism_sugar
        self.metabolism_spice=metabolism_spice
        self.vision=vision
    
    def get_sugar(self, pos):
        '''
        Used in self.get_sugar_amount()
        '''
        this_cell = self.model.grid.get_cell_list_contents(pos)
        for agent in this_cell:
            if type(agent) is Sugar:
                return agent
        return None

    def get_sugar_amount(self,pos):
        '''
        Used in self.move() as part of self.calculate_welfare()
        '''
        sugar_patch = self.get_sugar(pos)
        if sugar_patch:
            return sugar_patch.amount
        return 0
    
    def get_spice(self, pos):
        '''
        Used in self.get_spice_amount()
        '''
        this_cell = self.model.grid.get_cell_list_contents(pos)
        for agent in this_cell:
            if type(agent) is Spice:
                return agent
        return None
    
    def get_spice_amount(self,pos):
        '''
        Used in self.move() as part of self.calculate_welfare()
        '''
        spice_patch = self.get_spice(pos)
        if spice_patch:
            return spice_patch.amount
        return 0

    def is_occupied_by_other(self, pos):
        '''
        Helper function part 1 of self.move()
        '''
        if pos == self.pos:
            # agent's position is considered unoccupied and agent can stay there
            return False
        # Get contents of each cell in neighborhood
        this_cell = self.model.grid.get_cell_list_contents(pos)
        for a in this_cell:
            # see if occupied by another agent
            if isinstance(a, Trader):
                return True
        return False

    def calculate_welfare(self, sugar, spice):
        '''
        Helper function part 2 self.move()
        The higher the return, the more valuable that particular cell is 
        '''
        # Calculate total resources
        m_total = self.metabolism_sugar + self.metabolism_spice
        # Cobb Douglas functional form
        return sugar**(self.metabolism_sugar/m_total) * spice**(self.metabolism_spice/m_total)

    def move(self):
        '''
        Function for trader agent to identify optimal move for each step in 4 parts
        1 - identify all the possible moves
        2 - determine which move maximize welfare
        3 - find closest best option
        4 - move
        '''

        # 1. Identify all possible moves

        neighbors = [i for i in self.model.grid.get_neighborhood(
            pos = self.pos,
            moore = self.moore,
            include_center = True,
            radius = self.vision
        ) if not self.is_occupied_by_other(i)]

        # 2. Determine which move maximizes welfare

        welfares = [
            self.calculate_welfare(
                self.sugar + self.get_sugar_amount(pos),
                self.spice + self.get_spice_amount(pos)) 
            for pos in neighbors
        ]

        # 3. Find closest best option

        # Find the highest welfare in welfares
        max_welfare = max(welfares)
        # Get the index of max welfare cells
        candidate_indices = [i for i in range(len(welfares)) 
                             if math.isclose(welfares[i], max_welfare, rel_tol=1e-02)
                             ]
        
        # Convert index to positions of those cells
        candidates = [neighbors[i] for i in candidate_indices]

        min_dist = min(get_distance(self.pos, pos) for pos in candidates)    



# Model Class

In [58]:
class SugarscapeG1mt(mesa.Model):
    '''
    A model class to manage Sugarscape with Traders (G1mt)
    from Growing Artificial Societies by Axtell and Epstein
    '''

    def __init__(self, width=50, height=50, initial_population=200, endowment_min=25, endowment_max=50, metabolism_min=1, metabolism_max=5, vision_min=1, vision_max=5):
        
        super().__init__()
        # Iniciate width and height of sugarscape
        self.width=width
        self.height=height
        # Iniciate population attributes
        self.initial_population=initial_population
        self.endowment_min=endowment_min
        self.endowment_max=endowment_max
        self.metabolism_min=metabolism_min
        self.metabolism_max=metabolism_max
        self.vision_min=vision_min
        self.vision_max=vision_max

        # Iniciate mesa grid class
        self.grid=mesa.space.MultiGrid(self.width, self.height, torus=False)

        # Read in Landscape file from supplementary material
        sugar_distribution=np.genfromtxt("sugar-map.txt")
        spice_distribution=np.flip(sugar_distribution, 1)
        # plt.imshow(spice_distribution, origin="lower")

        # Iniciate Scheduler -> track what order our agents are activated
        self.schedule = mesa.time.RandomActivationByType(self)

        # Instanciate each agent
        agent_id = 0
        for _,(x,y) in self.grid.coord_iter():
            max_sugar = sugar_distribution[x,y] # define the sugar from the sugar distribution
            if max_sugar > 0:
                sugar = Sugar(agent_id, self, (x,y), max_sugar) # we pass self and the 'self' will be translated into an agent 'model' when it goes to the Agent class
                self.grid.place_agent(sugar, (x,y))
                self.schedule.add(sugar)
                # print(self.schedule.agents_by_type[Sugar][agent_id])
                agent_id+=1
                # print(sugar.unique_id, sugar.pos, sugar.max_sugar)
        
            max_spice = spice_distribution[x,y]
            if max_spice > 0:
                spice = Spice(agent_id, self, (x,y), max_spice)
                self.grid.place_agent(spice, (x,y))
                self.schedule.add(spice)
                # print(self.schedule.agents_by_type[Spice][agent_id])
                agent_id+=1
        
        for i in range(self.initial_population):
            # Get agent position
            x = self.random.randrange(self.width)
            y = self.random.randrange(self.height)
            # See Growing Artificial Society p.108 for initialization
            # Give agents initial endowment
            sugar = int(self.random.uniform(self.endowment_min, self.endowment_max+1))
            spice = int(self.random.uniform(self.endowment_min, self.endowment_max+1))
            # Give agents initial metabolism
            metabolism_sugar = int(self.random.uniform(self.metabolism_min, self.metabolism_max+1))
            metabolism_spice = int(self.random.uniform(self.metabolism_min, self.metabolism_max+1))
            # Give agents vision
            vision = int(self.random.uniform(self.vision_min, self.vision_max+1))
            # Create Trader object
            trader = Trader(agent_id, 
                            self, 
                            (x,y), 
                            moore=False, 
                            sugar=sugar,
                            spice=spice,
                            metabolism_sugar=metabolism_sugar,
                            metabolism_spice=metabolism_spice,
                            vision=vision)
            # Place agent
            self.grid.place_agent(trader, (x,y))
            self.schedule.add(trader)
            agent_id +=1

    def step(self):
        '''
        Unique step function that does staged activation of sugar and spice and then randomly activates traders.
        '''

        for sugar in self.schedule._agents_by_type[Sugar]:
            sugar.step()
        for spice in self.schedule._agents_by_type[Spice]:
            spice.step()
        
        # Step trader agents
        # To account for agent death and removal we need a separate data structure to iterate
        trader_shuffle = list(self.schedule._agents_by_type[Trader])
        self.random.shuffle(trader_shuffle)

        for agent in trader_shuffle:
            agent.move()

        self.schedule.steps += 1 # Important for Data Collector to track the number of steps
        # print(self.schedule.steps, self.schedule.time)
    
    def run_model(self, step_count=1000):

        for i in range(step_count):
            self.step()
            




In [67]:
model = SugarscapeG1mt()
model.run_model(step_count=1)


[(1, 32), (2, 31), (2, 32), (2, 33), (3, 30), (3, 31), (3, 32), (3, 33), (3, 34), (4, 29), (4, 30), (4, 31), (4, 32), (4, 33), (4, 34), (4, 35), (5, 28), (5, 29), (5, 30), (5, 31), (5, 32), (5, 33), (5, 34), (5, 35), (5, 36), (6, 29), (6, 30), (6, 31), (6, 32), (6, 34), (6, 35), (7, 30), (7, 32), (7, 33), (7, 34), (8, 31), (8, 32), (8, 33), (9, 32)] [30, 34] [(6, 35), (7, 34)]


NameError: name 'stop' is not defined